In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt
from scipy.spatial.distance import jensenshannon
from scipy.special import softmax
from tqdm import tqdm


In [10]:
ts_tracks = pd.read_csv("data/zebrafish/ZSNS003_tracks.csv")

In [11]:
ts_tracks.head(20)

,track_id,t,z,y,x,id,parent_track_id,parent_id
0,1,1,92.0,439.0,437.0,2015514,-1,-1
1,1,2,92.0,440.0,435.0,3013903,-1,2015514
2,1,3,92.0,449.0,430.0,4014957,-1,3013903
3,1,4,92.0,452.0,427.0,5016176,-1,4014957
4,1,5,94.0,454.0,423.0,6004881,-1,5016176
5,1,6,93.0,459.0,423.0,7013701,-1,6004881
6,1,7,93.0,461.0,420.0,8010934,-1,7013701
7,1,8,94.0,464.0,418.0,9004722,-1,8010934
8,1,9,94.0,466.0,418.0,10009982,-1,9004722
9,1,10,95.0,468.0,417.0,11012261,-1,10009982


In [37]:
children_dict = defaultdict(set)
parent_mapping = {}

In [38]:
for i, row in tqdm(ts_tracks.iterrows(), total=len(ts_tracks)):
    if row["parent_track_id"] == -1:
        continue
    children_dict[row["parent_track_id"]].add(row["track_id"])
    parent_mapping[row["track_id"]] = row["parent_track_id"]

100%|██████████| 4057611/4057611 [01:19<00:00, 51173.81it/s]


In [39]:
cell_groups = ts_tracks.groupby("track_id")

In [40]:
one_child_parents = set()
two_children_parents = set()
children_set = set()
for key, value in children_dict.items():
    for child in value:
        children_set.add(int(child))
    if len(value) == 1:
        one_child_parents.add(int(key))
    elif len(value) == 2:
        two_children_parents.add(int(key))

In [42]:
terminal_children = list(children_set - one_child_parents - two_children_parents)

In [55]:
terminal_children_coordinates = []
terminal_parents_coordinates = []
for child in tqdm(terminal_children):
    parent = parent_mapping[child]
    child_coordinates = cell_groups.get_group(child).sort_values(by="t").tail(1)[["x","y","z"]].values.tolist()[0]
    parent_coordinates = cell_groups.get_group(parent).sort_values(by="t").tail(1)[["x","y","z"]].values.tolist()[0]
    terminal_children_coordinates.append(child_coordinates)
    terminal_parents_coordinates.append(parent_coordinates)

100%|██████████| 26293/26293 [00:20<00:00, 1276.68it/s]


In [61]:
from multiprocessing import Pool
import multiprocessing as mp

def compute_cost_row(args):
    i, child_coords, parent_coords_list = args
    row = np.zeros(len(parent_coords_list))
    child_array = np.array(child_coords)
    for j, parent_coords in enumerate(parent_coords_list):
        row[j] = np.linalg.norm(child_array - np.array(parent_coords))
    return i, row

def compute_cost_matrix_parallel(terminal_children_coords, terminal_parents_coords, n_processes=None):
    if n_processes is None:
        n_processes = mp.cpu_count() - 1
    
    # Prepare arguments for each process
    args = [(i, terminal_children_coords[i], terminal_parents_coords) 
            for i in range(len(terminal_children_coords))]
    
    cost_matrix = np.zeros((len(terminal_children_coords), len(terminal_parents_coords)))
    
    with Pool(n_processes) as pool:
        results = list(tqdm(pool.imap(compute_cost_row, args), 
                           total=len(terminal_children_coords)))
    
    for i, row in results:
        cost_matrix[i] = row
    
    return cost_matrix

cost_matrix = compute_cost_matrix_parallel(terminal_children_coordinates, terminal_parents_coordinates)

100%|██████████| 26293/26293 [03:51<00:00, 113.61it/s]


In [62]:
cost_matrix.diagonal().sum()

np.float64(661278.6033251542)

In [58]:
row_ind, col_ind = linear_sum_assignment(cost_matrix)
total_cost = cost_matrix[row_ind, col_ind].sum()

In [59]:
total_cost

np.float64(373143.68440076755)

In [60]:
total_cost / cost_matrix.diagonal().sum()

np.float64(0.5642760593257709)